In [11]:
import pandas as pd

df = pd.read_csv("data/spamassassin_raw.csv")
df.head()

,text,label
0,Re: New Sequences Window\n Date: Wed...,ham
1,[zzzzteana] RE: Alexander\nMartin A posted:\nT...,ham
2,[zzzzteana] Moscow bomber\nMan Threatens Explo...,ham
3,[IRR] Klez: The Virus That Won't Die\nKlez: T...,ham
4,Re: [zzzzteana] Nothing like mama used to make...,ham


In [12]:
df.info()
print(df.describe())

<class 'pandas.DataFrame'>
RangeIndex: 6046 entries, 0 to 6045
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    6046 non-null   str  
 1   label   6046 non-null   str  
dtypes: str(2)
memory usage: 12.2 MB
                                                     text label
count                                                6046  6046
unique                                               5924     2
top     The database that Bill Gates doesnt want you t...   ham
freq                                                    5  4150


#### No missing values #####

In [13]:
print(df.shape)
print(df.isnull().sum())

(6046, 2)
text     0
label    0
dtype: int64


In [14]:
df["label"].value_counts()

label
ham     4150
spam    1896
Name: count, dtype: int64

## No blank emails ##

In [15]:
blank_emails = df["text"].fillna("").str.strip().eq("")
print("Blank emails:", blank_emails.sum())

Blank emails: 0


### 122 same texts ###

In [16]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 122


In [17]:
df["word_count"] = df["text"].str.split().str.len()
df.groupby("label")["word_count"].agg(["median", "mean", "max"]).round(1)

,median,mean,max
label,,,
ham,149.0,273.0,14959
spam,160.0,326.6,11625


In [23]:
clean_df = df[["text", "label"]].copy()

clean_df["text_key"] = (
    clean_df["text"]
    .str.casefold()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

clean_df = clean_df.drop_duplicates(
    subset=["text_key", "label"],
    ignore_index=True
)

labels_per_text = clean_df.groupby("text_key")["label"].nunique()
print("Texts with conflicting labels:", (labels_per_text > 1).sum())

clean_df = clean_df.drop(columns="text_key")

print("Before Cleaning:", df.shape)
print("After Cleaning:", clean_df.shape)

Texts with conflicting labels: 0
Before Cleaning: (6046, 3)
After Cleaning: (5916, 2)


In [24]:
clean_df.to_csv("data/spamassassin_clean.csv", index=False)
clean_df["label"].value_counts()

label
ham     4108
spam    1808
Name: count, dtype: int64